# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gunner2033d/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*My lane is Ranking*

*The goal is to prioritize content pages for human review when the content team has limited time. The model should help order pages from higher review priority to lower review priority using observable page-level signals such as impressions, CTR, average position, content age, and recent performance trends. Ranking fits because the final business action is a prioritized list of pages to review rather than a simple yes/no classification.*


In [5]:
import pandas as pd

df = pd.read_csv("/content/flyrank-internship/data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


## 2. Target or proxy

*Target/proxy: current performance decline.*

*The dataset does not contain a directly observed future refresh outcome, so I will use a rule-based proxy for this task. A page is marked as a positive case when its observed trend_direction is "down"; otherwise it is treated as a negative case. This proxy represents pages currently showing a decline in performance and can be used to explore whether page-level signals can prioritize pages for review.*

*This is a proxy, not proof that a page will decline in the future or that a refresh will improve it.*

In [6]:
df["decline_proxy"] = (df["trend_direction"] == "down").astype(int)

print(df["decline_proxy"].value_counts())
print(df["decline_proxy"].value_counts(normalize=True).round(3))
df['trend_direction'].value_counts()


decline_proxy
1    16262
0    13738
Name: count, dtype: int64
decline_proxy
1    0.542
0    0.458
Name: proportion, dtype: float64


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152


## 3. Success metric

*Success metric: Precision@K.*

*Precision@K measures how many of the pages in the model's top-K recommendations are actually positive proxy cases. This matches the business decision because the content team has limited time and may only review the highest-priority pages. A good model should therefore place a high proportion of relevant pages near the top of the review queue.*

In [7]:
K = 50

print(f"Primary metric: Precision@{K}")


Primary metric: Precision@50


## 4. The unit of analysis, as a real dataframe

*One row represents one content-page observation.*

*Each row contains anonymized content attributes and search-performance signals for a single page, such as impressions, clicks, CTR, average position, content age, and trend information. The model will use these page-level observations to prioritize which pages should be reviewed first.*

In [9]:
sample = df.sample(5, random_state=42)
display(sample)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,decline_proxy
2308,content_9824710082d8,client_3fdba35f04,0.0,0.0,LOW,0.00,keyword article,informational,1397.0,9273.0,...,0.00,21.3,0.00,0.00,0.0,low,page_3_5,stable,8.0,0
22404,content_3efa3a7c46bb,client_f74efabef1,0.0,0.0,LOW,0.00,keyword article,informational,3188.0,22026.0,...,0.09,8.8,4.35,4.35,0.0,good,page_1,up,22.9,0
23397,content_575dc8a2ab0f,client_25fc0e7096,NaN,NaN,NaN,NaN,feedly article,NaN,3381.0,21992.0,...,0.00,0.0,0.00,20.83,0.0,low,top_3,new,NaN,0
25058,content_0dbd6911ba04,client_d029fa3a95,0.0,0.0,LOW,0.00,comparison article,informational,2892.0,19212.0,...,0.00,8.1,0.00,40.00,0.0,low,page_1,down,-57.4,1
2664,content_bbaf87019afb,client_19581e27de,30.0,1.0,HIGH,1.52,keyword article,commercial,NaN,NaN,...,0.23,30.9,2.33,2.08,0.0,good,page_3_5,down,-20.8,1


## 5. Why ML beats a fixed rule here

*A fixed rule based on one signal, such as CTR below a threshold, may miss pages that have different combinations of visibility, position, age, and engagement. ML can consider multiple signals together when identifying pages that may deserve higher review priority. The model should still be treated as decision support and compared with a simple baseline.*

In [11]:
cols = [
    "ctr",
    "avg_position",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "trend_direction"
]

display(df[cols].sample(5, random_state=42))

,ctr,avg_position,content_age_days,impressions_90d,clicks_90d,trend_direction
2308,0.00,21.3,174,283,0,stable
22404,0.09,8.8,134,8878,8,up
23397,0.00,0.0,109,3,0,new
25058,0.00,8.1,151,124,0,down
2664,0.23,30.9,466,4294,10,down


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.